# Evaluation & Serving the Fine-Tuned Model — Week 5

**Notebook:** `07_evaluation_and_serving.ipynb`  
**Estimated time:** 30 minutes  

## Objectives
1. Use LLM-as-Judge (Claude-haiku) to score base vs fine-tuned model responses on resume Q&A
2. Run a GSM8K micro-benchmark to measure math reasoning capability
3. Benchmark inference latency
4. Merge the LoRA adapter, convert to GGUF, and serve via Ollama

## Prerequisites
- `outputs/sft_adapter/` — fine-tuned LoRA adapter from NB05
- `outputs/synthetic_dataset.json` — synthetic Q&A data from NB03
- `FINETUNE_BACKEND` env var set to `"mlx"` (Path A) or `"hf"` (Path B)

In [4]:
import sys
import importlib
import os

sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv(override=True)

%matplotlib inline

from src.cost_tracker import CostTracker
tracker = CostTracker()

FINETUNE_BACKEND = os.getenv("FINETUNE_BACKEND", "hf")  # "mlx" or "hf"
print(f"Fine-tune backend: {FINETUNE_BACKEND}")
print("Setup complete.")

Fine-tune backend: hf
Setup complete.


---
## Part 1: LLM-as-Judge Evaluation

How do we know if fine-tuning improved anything? We use **Claude-haiku as a judge** with a 1-5 rubric.
The judge sees both a question and a model answer, then scores it — no ground-truth labels needed.

### Scoring Rubric
```
Score 1: Answer is factually wrong or completely irrelevant
Score 2: Answer is vague, missing key details
Score 3: Answer is mostly correct but generic
Score 4: Answer is correct, specific, and well-written
Score 5: Answer is exceptional — specific facts, confident tone, perfectly formatted
```

We build a 5-question test set covering the main resume dimensions, then compare base vs fine-tuned scores.

In [5]:
import importlib
import src.model_eval as _me
importlib.reload(_me)
import src.sft_trainer as _st
importlib.reload(_st)

from src.model_eval import judge_llm_eval, save_scoreboard
from src.sft_trainer import SFTRunner
from src.llm_client import LLMClient

# Claude-haiku as judge (Path A = Claude API)
judge_client = LLMClient(path="A")

rubric_text = """\
Score 1: Answer is factually wrong or completely irrelevant
Score 2: Answer is vague, missing key details
Score 3: Answer is mostly correct but generic
Score 4: Answer is correct, specific, and well-written
Score 5: Answer is exceptional — specific facts, confident tone, perfectly formatted
"""

# 5-question test set covering key resume dimensions
test_set = [
    {
        "question": "Where did Scott complete his undergraduate education and what did he study?",
        "category": "education"
    },
    {
        "question": "What programming languages and frameworks does Scott have experience with?",
        "category": "skills"
    },
    {
        "question": "Describe Scott's most recent work experience and his key responsibilities.",
        "category": "experience"
    },
    {
        "question": "What is one notable project or achievement Scott is proud of from his career?",
        "category": "achievements"
    },
    {
        "question": "What kind of roles or opportunities is Scott currently looking for?",
        "category": "goals"
    },
]

print(f"Test set: {len(test_set)} questions")
for i, item in enumerate(test_set, 1):
    print(f"  Q{i} [{item['category']}]: {item['question'][:60]}...")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
Test set: 5 questions
  Q1 [education]: Where did Scott complete his undergraduate education and wha...
  Q2 [skills]: What programming languages and frameworks does Scott have ex...
  Q3 [experience]: Describe Scott's most recent work experience and his key res...
  Q4 [achievements]: What is one notable project or achievement Scott is proud of...
  Q5 [goals]: What kind of roles or opportunities is Scott currently looki...


In [6]:
# Load fine-tuned SFT adapter
runner = SFTRunner(backend=FINETUNE_BACKEND)
runner.load_adapter("../outputs/sft_adapter")  # load existing fine-tuned adapter

model_fn = lambda prompt: runner.generate(prompt, max_new_tokens=150)

print("Fine-tuned model_fn ready.")
# Quick sanity check
sample_response = model_fn("What programming languages does Scott know?")
print(f"Sample response: {sample_response[:150]}")

[sft_trainer] Initialized SFTRunner backend=hf model=Qwen/Qwen2.5-0.5B-Instruct
[sft_trainer] Adapter path set to ../outputs/sft_adapter; will be loaded on next generate() call.
Fine-tuned model_fn ready.
[sft_trainer] No model in memory. Loading from adapter path...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[sft_trainer] Generating response for prompt: What programming languages does Scott know?...
[sft_trainer] Generation complete (797 chars)
Sample response:  Scott has experience with Python, R, and SQL. He also knows JavaScript and TypeScript. Scott is currently learning Python because he wants to build M


In [7]:
# Run LLM-as-Judge evaluation
print("Running LLM-as-Judge evaluation (fine-tuned model)...")
results = judge_llm_eval(
    model_fn=model_fn,
    test_set=test_set,
    rubric=rubric_text,
    judge_client=judge_client,
)

save_scoreboard(results, "../outputs/eval_scoreboard.json")
print("Scoreboard saved to outputs/eval_scoreboard.json")

Running LLM-as-Judge evaluation (fine-tuned model)...
[model_eval] Starting LLM-as-judge eval on 5 items with model=claude-haiku-4-5-20251001
[model_eval] Evaluating item 1/5: Where did Scott complete his undergraduate education and wha...
[sft_trainer] Generating response for prompt: Where did Scott complete his undergraduate education and what did he study?...
[sft_trainer] Generation complete (807 chars)
[model_eval] Judge call failed on item 1: 'dict' object has no attribute 'strip'
[model_eval] Item 1 score: 3/5
[model_eval] Evaluating item 2/5: What programming languages and frameworks does Scott have ex...
[sft_trainer] Generating response for prompt: What programming languages and frameworks does Scott have experience with?...
[sft_trainer] Generation complete (638 chars)
[model_eval] Judge call failed on item 2: 'dict' object has no attribute 'strip'
[model_eval] Item 2 score: 3/5
[model_eval] Evaluating item 3/5: Describe Scott's most recent work experience and his key res...

In [8]:
# Print score comparison table
details = results.get("details", []) if isinstance(results, dict) else []
print(f"{'#':<4} {'Score':>6} {'Question':<60}")
print("-" * 75)
for i, item in enumerate(details, 1):
    score = item.get("score", "N/A")
    q = item.get("question", "")[:60]
    print(f"{i:<4} {str(score):>6} {q:<60}")

print("-" * 75)
print(f"{'MEAN':<4} {results.get('mean', 0):>6.2f}")
print(f"{'PASS':<4} {results.get('pass_rate', 0)*100:>5.0f}%  (score >= 3)")


#     Score Question                                                    
---------------------------------------------------------------------------
1         3 Where did Scott complete his undergraduate education and wha
2         3 What programming languages and frameworks does Scott have ex
3         3 Describe Scott's most recent work experience and his key res
4         3 What is one notable project or achievement Scott is proud of
5         3 What kind of roles or opportunities is Scott currently looki
---------------------------------------------------------------------------
MEAN   3.00
PASS   100%  (score >= 3)


---
## Part 2: GSM8K Micro-Benchmark

GSM8K is a dataset of grade-school math word problems. We use 5 samples to check whether our model preserved (or degraded) general reasoning during fine-tuning.

> **Expected result:** Our model was fine-tuned on resume Q&A data — not math. Expect **0–10% accuracy**. This establishes the baseline. If you ran the GRPO RL bonus in NB06, that technique specifically targets reasoning improvement.

In [9]:
import importlib
import src.model_eval as _me
importlib.reload(_me)
from src.model_eval import gsm8k_micro_eval

print("Running GSM8K micro-benchmark (n=5 samples)...")
gsm_results = gsm8k_micro_eval(model_fn, n=5)

print(f"\nGSM8K accuracy: {gsm_results['accuracy']:.0%} ({gsm_results['correct']}/{gsm_results['total']})")
print()
print("Note: Our model was not trained for math — 0-10% is expected.")
print("GRPO RL fine-tuning (NB06 bonus) would target this metric directly.")

Running GSM8K micro-benchmark (n=5 samples)...
[model_eval] GSM8K micro-eval: running 5 problems...
[model_eval] Problem 1/5: If a store has 48 apples and sells 13, how many remain? Show...
[sft_trainer] Generating response for prompt: If a store has 48 apples and sells 13, how many remain? Show work then #### answ...
[sft_trainer] Generation complete (423 chars)
[model_eval] Problem 1: WRONG (expected=35)
[model_eval] Problem 2/5: A car travels 60 mph for 2.5 hours. How many miles? Show wor...
[sft_trainer] Generating response for prompt: A car travels 60 mph for 2.5 hours. How many miles? Show work then #### answer....
[sft_trainer] Generation complete (227 chars)
[model_eval] Problem 2: WRONG (expected=150)
[model_eval] Problem 3/5: Sam earns $15/hr and works 8 hrs/day for 5 days. Total pay? ...
[sft_trainer] Generating response for prompt: Sam earns $15/hr and works 8 hrs/day for 5 days. Total pay? Show work then #### ...
[sft_trainer] Generation complete (495 chars)
[model_eval] P

---
## Part 3: Latency Benchmark

Before serving to users, we need to know how fast the model responds. We measure:
- **Mean latency** — average response time across N calls
- **P95 latency** — the 95th-percentile response time (what 95% of users experience or better)

In [10]:
import importlib
import src.model_eval as _me
importlib.reload(_me)
from src.model_eval import latency_benchmark

print("Running latency benchmark (n_calls=3)...")
lat = latency_benchmark(model_fn, n_calls=3)

print(f"Mean latency: {lat['mean_ms']:.0f}ms | P95: {lat['p95_ms']:.0f}ms")
print(f"Min: {lat.get('min_ms', 'N/A'):.0f}ms | Max: {lat.get('max_ms', 'N/A'):.0f}ms")

Running latency benchmark (n_calls=3)...
[model_eval] Latency benchmark: 3 calls...
[model_eval] Call 1/3...
[sft_trainer] Generating response for prompt: What is the capital of France?...
[sft_trainer] Generation complete (652 chars)
[model_eval] Call 1: 8122.3 ms
[model_eval] Call 2/3...
[sft_trainer] Generating response for prompt: Explain what gradient descent is in one sentence....
[sft_trainer] Generation complete (884 chars)
[model_eval] Call 2: 7984.1 ms
[model_eval] Call 3/3...
[sft_trainer] Generating response for prompt: Write a Python function that returns the factorial of n....
[sft_trainer] Generation complete (663 chars)
[model_eval] Call 3: 7940.5 ms
[model_eval] Benchmark: min=7940.5ms, mean=8015.6ms, p50=7984.1ms, p95=8122.3ms, max=8122.3ms, tok/s=13.8
Mean latency: 8016ms | P95: 8122ms
Min: 7940ms | Max: 8122ms


---
## Part 4: Serving via Ollama

To deploy our fine-tuned model in production (or for local demos), we:
1. **Merge** the LoRA adapter weights into the base model (creates a standalone model)
2. **Convert** to GGUF format (llama.cpp's efficient quantized format for CPU/GPU inference)
3. **Create a Modelfile** (Ollama's configuration format)
4. **Register and test** with Ollama

If `llama.cpp` is not installed, the notebook will print instructions and continue gracefully — you can still use the merged (non-quantized) model.

In [11]:
import importlib
import src.model_serve as _ms
importlib.reload(_ms)
from src.model_serve import merge_lora, convert_to_gguf, make_ollama_modelfile, ollama_create_and_test

# Step 1: Merge LoRA adapter into the base model
print("Step 1: Merging LoRA adapter into base model...")
merged_path = merge_lora(
    base_model_path="Qwen/Qwen2.5-0.5B-Instruct",
    adapter_path="../outputs/sft_adapter",
    output_path="../outputs/merged_model",
)
print(f"Merged model saved to: {merged_path}")


Step 1: Merging LoRA adapter into base model...
[model_serve] Merging LoRA adapter into base model...
[model_serve] Base: Qwen/Qwen2.5-0.5B-Instruct
[model_serve] Adapter: ../outputs/sft_adapter
[model_serve] Output: ../outputs/merged_model
[model_serve] Loading base model with dtype=float16...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[model_serve] Loading LoRA adapter (HF PEFT format)...
[model_serve] Merging and unloading LoRA weights...
[model_serve] Saving merged model to ../outputs/merged_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[model_serve] Merge complete -> ../outputs/merged_model
Merged model saved to: ../outputs/merged_model


In [13]:
# Step 2: Convert to GGUF (Q4_K_M quantization)
# Graceful fallback if llama.cpp is not installed
print("Step 2: Converting to GGUF (Q4_K_M)...")
print("Note: Requires llama.cpp. If not installed, will print instructions and skip.")

gguf_path = convert_to_gguf(
    hf_model_path=merged_path,
    output_path="../outputs/hw5_finetuned.Q4_K_M.gguf",
)

if gguf_path:
    print(f"GGUF file saved to: {gguf_path}")
else:
    print("GGUF conversion skipped. Will use merged model directory for Ollama.")
    print()
    print("To install llama.cpp for GGUF conversion:")
    print("  brew install llama.cpp   # macOS")
    print("  # or build from source: https://github.com/ggerganov/llama.cpp")


Step 2: Converting to GGUF (Q4_K_M)...
Note: Requires llama.cpp. If not installed, will print instructions and skip.
[model_serve] Converting ../outputs/merged_model to GGUF (quant=Q4_K_M)...
[model_serve] Found llama.cpp convert script: /home/jovyan/llama.cpp/convert_hf_to_gguf.py
[model_serve] Step 1: Converting to F16 GGUF...
[model_serve] Conversion output:

[model_serve] Step 2: Quantizing to Q4_K_M...
[model_serve] Quantization output:

main: quantize time = 10114.55 ms
main:    total time = 10114.55 ms

[model_serve] Removed intermediate F16 file
[model_serve] GGUF ready at: ../outputs/hw5_finetuned.Q4_K_M.gguf/merged_model-Q4_K_M.gguf
GGUF file saved to: ../outputs/hw5_finetuned.Q4_K_M.gguf/merged_model-Q4_K_M.gguf


In [14]:
# Step 3: Generate Ollama Modelfile
print("Step 3: Generating Ollama Modelfile...")

modelfile_content = make_ollama_modelfile(
    gguf_path=gguf_path or merged_path,
    output_path="../outputs/ollama_modelfile.txt",
    system_prompt="You are a helpful assistant trained on resume Q&A data.",
)

print("=== Modelfile content ===")
print(modelfile_content)


Step 3: Generating Ollama Modelfile...
[model_serve] Generating Ollama Modelfile for hw5-finetuned...
[model_serve] Modelfile saved to ../outputs/ollama_modelfile.txt
[model_serve] To register with Ollama: ollama create hw5-finetuned -f ../outputs/ollama_modelfile.txt
=== Modelfile content ===
FROM /workspace/Homework5-Submission/outputs/hw5_finetuned.Q4_K_M.gguf/merged_model-Q4_K_M.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

SYSTEM """
You are a helpful assistant trained on resume Q&A data.
"""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 2048
PARAMETER num_predict 300
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|endoftext|>"



In [15]:
import importlib                                                                                                                        
import src.model_serve                                                                                                                  
importlib.reload(src.model_serve)                                                                                                       
importlib.reload(src.model_serve)
from src.model_serve import ollama_create_and_test

In [16]:
# Step 4: Create and test Ollama model
print("Step 4: Creating Ollama model 'hw5-finetuned'...")
print("Note: Requires 'ollama' to be running. Start it with: ollama serve")

response = ollama_create_and_test(
    model_name="hw5-finetuned",
    modelfile_path="../outputs/ollama_modelfile.txt",
)
print(response)

Step 4: Creating Ollama model 'hw5-finetuned'...
Note: Requires 'ollama' to be running. Start it with: ollama serve
[model_serve] Creating Ollama model 'hw5-finetuned' from ../outputs/ollama_modelfile.txt...
[model_serve] Ollama version: ollama version is 0.23.0
[model_serve] Running: ollama create hw5-finetuned -f ../outputs/ollama_modelfile.txt
[model_serve] ollama create output:

[model_serve] Testing model with prompt: Who are you and what can you help me with?...
[model_serve] (first inference loads the model — may take up to 5 min on Mac)
[model_serve] Ollama response: As an AI language model, I am here to assist you with any questions you may have. Please feel free to ask anything you need help with.
As an AI language model, I am here to assist you with any questions you may have. Please feel free to ask anything you need help with.


---
## TODO 1: Baseline Comparison

Run the same LLM-as-judge evaluation on a **baseline** (non-fine-tuned) model — e.g., the Ollama `qwen2.5:0.5b` base or `qwen3.5:27b` — using the same 5 test questions.

Then compare:
- Baseline average score vs fine-tuned average score
- Which categories improved the most?
- Did fine-tuning help? Where did it hurt (if anywhere)?

**Starter code:**

In [18]:
# TODO 1: Run baseline evaluation and compare
# Hint: define a baseline_model_fn using LLMClient(path="B") or ollama

# Example:
# from src.llm_client import LLMClient
# base_client = LLMClient(path="B")  # Ollama
# baseline_model_fn = lambda prompt: base_client.generate(prompt)["content"]
#
# baseline_results = judge_llm_eval(
#     model_fn=baseline_model_fn,
#     test_set=test_set,
#     rubric=rubric_text,
#     judge_client=judge_client,
# )
# save_scoreboard(baseline_results, "outputs/eval_scoreboard_baseline.json")
#
# Compare:
# base_avg = sum(r["score"] for r in baseline_results) / len(baseline_results)
# ft_avg = sum(r["score"] for r in results) / len(results)
# print(f"Baseline avg: {base_avg:.2f} | Fine-tuned avg: {ft_avg:.2f} | Delta: {ft_avg - base_avg:+.2f}")

  # TODO 1: Run baseline evaluation and compare

def content_only(response):
    if isinstance(response, dict):
        return response.get("content") or response.get("error", "")
    return str(response)

class TextOnlyJudge:
    def __init__(self, client):
        self.client = client

    def generate(self, *args, **kwargs):
        return content_only(self.client.generate(*args, **kwargs))

judge_text_client = TextOnlyJudge(judge_client)

# Re-run fine-tuned eval with corrected judge response handling
print("Running corrected fine-tuned evaluation...")
ft_results = judge_llm_eval(
    model_fn=model_fn,
    test_set=test_set,
    rubric=rubric_text,
    judge_client=judge_text_client,
)

save_scoreboard(ft_results, "../outputs/eval_scoreboard.json")

# Baseline: use Ollama non-fine-tuned model
base_client = LLMClient(path="B")
available_models = base_client.get_available_models()

preferred_baseline = "qwen2.5:0.5b"
baseline_model = preferred_baseline if preferred_baseline in available_models else base_client.default_model

print(f"Using baseline model: {baseline_model}")

baseline_model_fn = lambda prompt: content_only(
    base_client.generate(
        prompt,
        model=baseline_model,
        temperature=0.2,
        max_tokens=200,
    )
)

print("Running baseline evaluation...")
baseline_results = judge_llm_eval(
    model_fn=baseline_model_fn,
    test_set=test_set,
    rubric=rubric_text,
    judge_client=judge_text_client,
)

save_scoreboard(baseline_results, "../outputs/eval_scoreboard_baseline.json")

# Compare averages
base_avg = baseline_results["mean"]
ft_avg = ft_results["mean"]
delta = ft_avg - base_avg

print(f"Baseline avg: {base_avg:.2f}")
print(f"Fine-tuned avg: {ft_avg:.2f}")
print(f"Delta: {delta:+.2f}")

# Compare by category
category_rows = []
for item, base_detail, ft_detail in zip(test_set, baseline_results["details"], ft_results["details"]):
    category_rows.append({
        "category": item["category"],
        "baseline": base_detail["score"],
        "fine_tuned": ft_detail["score"],
        "delta": ft_detail["score"] - base_detail["score"],
    })

print("\nCategory comparison:")
for row in category_rows:
    print(
        f"{row['category']}: "
        f"baseline={row['baseline']}, "
        f"fine_tuned={row['fine_tuned']}, "
        f"delta={row['delta']:+d}"
    )

max_delta = max(row["delta"] for row in category_rows)
best_categories = [row["category"] for row in category_rows if row["delta"] == max_delta]
hurt_categories = [row["category"] for row in category_rows if row["delta"] < 0]

if delta > 0:
    overall = "Fine-tuning helped overall."
elif delta < 0:
    overall = "Fine-tuning hurt overall."
else:
    overall = "Fine-tuning was neutral overall."

hurt_text = (
    f"It hurt in: {', '.join(hurt_categories)}."
    if hurt_categories
    else "It did not hurt any category."
)

todo1_reflection = (
    f"Baseline average score was {base_avg:.2f}, while the fine-tuned model averaged {ft_avg:.2f}, "
    f"for a delta of {delta:+.2f}. The largest improvement was in "
    f"{', '.join(best_categories)} ({max_delta:+d}). {overall} {hurt_text}"
)

print("\nTODO 1 reflection:")
print(todo1_reflection)

Running corrected fine-tuned evaluation...
[model_eval] Starting LLM-as-judge eval on 5 items with model=claude-haiku-4-5-20251001
[model_eval] Evaluating item 1/5: Where did Scott complete his undergraduate education and wha...
[sft_trainer] Generating response for prompt: Where did Scott complete his undergraduate education and what did he study?...
[sft_trainer] Generation complete (874 chars)
[model_eval] Item 1 score: 1/5
[model_eval] Evaluating item 2/5: What programming languages and frameworks does Scott have ex...
[sft_trainer] Generating response for prompt: What programming languages and frameworks does Scott have experience with?...
[sft_trainer] Generation complete (726 chars)
[model_eval] Item 2 score: 4/5
[model_eval] Evaluating item 3/5: Describe Scott's most recent work experience and his key res...
[sft_trainer] Generating response for prompt: Describe Scott's most recent work experience and his key responsibilities....
[sft_trainer] Generation complete (808 chars)
[m

---
## TODO 2: What is Quantization?

In 2-3 sentences, explain:
1. What **quantization** means for LLMs (reducing weight precision from float32/float16 to 4-bit integers)
2. How `bitsandbytes` 4-bit (used in NB04 for QLoRA) and **GGUF Q4_K_M** (used here) are related
3. Why `Q4_K_M` specifically — what does "K_M" mean, and what's the quality/speed tradeoff?

Write your answer in the cell below:

**Your answer (TODO 2):**

Quantization reduces an LLM’s memory and compute cost by storing model weights at lower precision, such as 4-bit integers instead of float16 or float32. The bitsandbytes 4-bit setup used earlier is mainly for training-time QLoRA, while GGUF Q4_K_M is a deployment format used by llama.cpp/Ollama for efficient local inference. Q4_K_M uses mixed “K-quants,” keeping some parts of the model at slightly higher precision than plain Q4_0, so it usually gives a better quality/speed tradeoff for serving.

---
## Summary

In [19]:
import json
import os
from datetime import datetime

# Summarize outputs
outputs = [
    "../outputs/eval_scoreboard.json",
    "../outputs/merged_model/",
    "../outputs/ollama_modelfile.txt",
]
print("=== NB07 Outputs ===")
for path in outputs:
    exists = os.path.exists(path)
    print(f"  {'[OK]' if exists else '[MISSING]'} {path}")

# Append to reflection log
def append_to_reflection(nb_id: str, nb_title: str, reflection: str, path: str = "../outputs/reflection_log.json"):
    log = []
    if os.path.exists(path):
        with open(path) as f:
            log = json.load(f)
    log.append({
        "notebook": nb_id,
        "title": nb_title,
        "reflection": reflection,
        "timestamp": datetime.now().isoformat(),
    })
    with open(path, "w") as f:
        json.dump(log, f, indent=2)
    print(f"Reflection appended to {path}")

append_to_reflection(
    "07",
    "Evaluation & Serving",
    todo1_reflection if 'todo1_reflection' in dir() else "[TODO 1 not completed]",
)

tracker.report()

=== NB07 Outputs ===
  [OK] ../outputs/eval_scoreboard.json
  [OK] ../outputs/merged_model/
  [OK] ../outputs/ollama_modelfile.txt
Reflection appended to ../outputs/reflection_log.json
API COST REPORT
Total API calls:     0
Total input tokens:  0
Total output tokens: 0
Total cost:          $0.0000

